# DESASTRES NATURALES ENTRE 1970-2021

PRUEBAS DE MEJORA DE LAS COORDENADAS CON GEOPY

Encontramos varios errores en las coordenadas de la base de datos (columnas Latitude y Longitude que pasan a ser Latitud_corregida y Longitud_corregida). 
Hemos revisado con la IA Gemini Code Assist posibles opciones pero no hemos podido solventar los problemas principalmente por falta de tiempo, pues el código que nos proporciona tarda muchas horas en obtener resultados, si los obtiene, y la mayor parte de las ocaciones modifica columnas y otras partes del dataset. 
Hemos preferido dejar ésta parte para Next steps


In [ ]:
import pandas as pd
import re

def corregir_coordenadas(coord):
    """
    Convierte una coordenada en formato de texto (ej. '30.37 N', '78.30 W')
    a un formato numérico decimal.
    - 'N' y 'E' son positivos.
    - 'S' y 'W' son negativos.
    """
    # Si la coordenada ya es un número o está vacía (NaN), no se hace nada.
    if pd.isna(coord) or isinstance(coord, (int, float)):
        return coord

    # Asegurarse de que la coordenada es una cadena de texto para procesarla.
    coord_str = str(coord).strip().upper()

    # Usar expresiones regulares para encontrar el número.
    numero_match = re.search(r'[-+]?\d*\.\d+|\d+', coord_str)
    if not numero_match:
        return None # Retorna None si no se encuentra un número válido.

    numero = float(numero_match.group(0))

    # Aplicar el signo correcto según la letra (N, S, E, W).
    if 'S' in coord_str or 'W' in coord_str:
        return -abs(numero)
    
    return abs(numero)

try:
    # 1. Cargar el archivo CSV en un DataFrame de pandas
    # Asegúrate de que la ruta del archivo sea la correcta.
    ruta_archivo = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_revisados.csv'
    df = pd.read_csv(ruta_archivo)

    # 2. Crear las nuevas columnas aplicando la función de corrección
    print("Corrigiendo las columnas 'Latitude' y 'Longitude'...")
    df['Latitud_corregida'] = df['Latitude'].apply(corregir_coordenadas)
    df['Longitud_corregida'] = df['Longitude'].apply(corregir_coordenadas)
    print("Corrección completada.")

    # 3. Mostrar una vista previa de las columnas originales y las corregidas
    print("\nVista previa de los datos corregidos:")
    print(df[['Latitude', 'Latitud_corregida', 'Longitude', 'Longitud_corregida']].head(10))
    
    # Ejemplo de una fila que necesitaba corrección (fila 206)
    print("\nEjemplo de una fila específica (índice 206):")
    print(df.loc[206, ['Latitude', 'Latitud_corregida', 'Longitude', 'Longitud_corregida']])

    # 4. Guardar el DataFrame con las nuevas columnas en un nuevo archivo CSV
    ruta_salida = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_corregidos.csv'
    df.to_csv(ruta_salida, index=False)
    print(f"\n¡Éxito! El archivo con las coordenadas corregidas se ha guardado en: {ruta_salida}")

except FileNotFoundError:
    print(f"Error: No se pudo encontrar el archivo en la ruta especificada.")
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")


Corrigiendo las columnas 'Latitude' y 'Longitude'...
Corrección completada.

Vista previa de los datos corregidos:
  Latitude  Latitud_corregida Longitude  Longitud_corregida
0      NaN                NaN       NaN                 NaN
1      NaN                NaN       NaN                 NaN
2      NaN                NaN       NaN                 NaN
3      NaN                NaN       NaN                 NaN
4      NaN                NaN       NaN                 NaN
5      NaN                NaN       NaN                 NaN
6      NaN                NaN       NaN                 NaN
7      NaN                NaN       NaN                 NaN
8      NaN                NaN       NaN                 NaN
9      NaN                NaN       NaN                 NaN

Ejemplo de una fila específica (índice 206):
Latitude               30.37 N
Latitud_corregida        30.37
Longitude             104.06 E
Longitud_corregida      104.06
Name: 206, dtype: object

¡Éxito! El archivo con las co

In [ ]:
import pandas as pd
import unicodedata
import re

def limpiar_titulos(df):
    """
    Limpia y estandariza los títulos de las columnas de un DataFrame.
    - Convierte a minúsculas.
    - Elimina acentos y caracteres especiales.
    - Reemplaza espacios y caracteres no alfanuméricos por guiones bajos.
    """
    nuevas_columnas = []
    for col in df.columns:
        # 1. Normalizar para descomponer acentos (ej. 'á' -> 'a' + '´')
        col_normalizada = unicodedata.normalize('NFKD', col)
        # 2. Codificar a ASCII ignorando los caracteres no ASCII (los acentos)
        col_ascii = col_normalizada.encode('ascii', 'ignore').decode('utf-8')
        # 3. Convertir a minúsculas y reemplazar espacios/caracteres no alfanuméricos
        col_limpia = re.sub(r'[^a-zA-Z0-9]+', '_', col_ascii).lower()
        # 4. Eliminar guiones bajos al principio o al final
        col_limpia = col_limpia.strip('_')
        nuevas_columnas.append(col_limpia)
    
    df.columns = nuevas_columnas
    return df

try:
    # --- Carga de datos ---
    ruta_archivo = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_corregidos.csv'
    df = pd.read_csv(ruta_archivo)

    # --- 1. Rellenar valores nulos en las coordenadas corregidas ---
    print("Rellenando valores nulos en 'Latitud_corregida' y 'Longitud_corregida'...")
    
    # Usamos .fillna() que es la forma más eficiente en pandas para esta tarea.
    # Rellena los nulos en 'Latitud_corregida' con los valores de 'Latitude'.
    df['Latitud_corregida'].fillna(df['Latitude'], inplace=True)
    
    # Rellena los nulos en 'Longitud_corregida' con los valores de 'Longitude'.
    df['Longitud_corregida'].fillna(df['Longitude'], inplace=True)
    
    print("Valores nulos rellenados.")

    # --- 2. Limpieza de los títulos de las columnas ---
    print("\nLimpiando los títulos de las columnas para mejor compatibilidad...")
    
    # Eliminar la columna 'Unnamed: 0' si existe
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
        print("Columna 'Unnamed: 0' eliminada.")
        
    # Aplicar la función de limpieza a todos los títulos
    df = limpiar_titulos(df)
    print("Títulos de columnas estandarizados.")
    print("\nNuevos títulos de las columnas:")
    print(df.columns.tolist())

    # --- Verificación y guardado ---
    print("\nVista previa de las columnas de coordenadas actualizadas:")
    # Mostramos las 4 columnas de coordenadas para verificar el relleno de nulos
    print(df[['latitude', 'latitud_corregida', 'longitude', 'longitud_corregida']].head(10))

    # Guardar el DataFrame final en un nuevo archivo CSV
    ruta_salida = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_finales_limpios.csv'
    df.to_csv(ruta_salida, index=False)
    
    print(f"\n¡Proceso completado! El archivo final ha sido guardado en:\n{ruta_salida}")

except FileNotFoundError:
    print(f"Error: No se pudo encontrar el archivo en la ruta: {ruta_archivo}")
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")

Rellenando valores nulos en 'Latitud_corregida' y 'Longitud_corregida'...
Valores nulos rellenados.

Limpiando los títulos de las columnas para mejor compatibilidad...
Columna 'Unnamed: 0' eliminada.
Títulos de columnas estandarizados.

Nuevos títulos de las columnas:
['dis_no', 'year', 'disaster_subgroup', 'disaster_type', 'disaster_subtype', 'event_name', 'country', 'region', 'continent', 'location', 'origin', 'associated_dis', 'dis_mag_value', 'dis_mag_scale', 'latitude', 'longitude', 'total_deaths', 'total_affected', 'start_date', 'end_date', 'duration', 'latitud_corregida', 'longitud_corregida']

Vista previa de las columnas de coordenadas actualizadas:
  latitude  latitud_corregida longitude  longitud_corregida
0      NaN                NaN       NaN                 NaN
1      NaN                NaN       NaN                 NaN
2      NaN                NaN       NaN                 NaN
3      NaN                NaN       NaN                 NaN
4      NaN                NaN    

C:\Users\krigu\AppData\Local\Temp\ipykernel_11972\2170880457.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Latitud_corregida'].fillna(df['Latitude'], inplace=True)
C:\Users\krigu\AppData\Local\Temp\ipykernel_11972\2170880457.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df['Latitud_corregida'].fillna(df['Lat


¡Proceso completado! El archivo final ha sido guardado en:
c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_finales_limpios.csv


In [43]:
df_finales = pd.read_csv("datos_finales_limpios.csv", index_col=0)
df_finales = df_finales.reset_index()
df_finales.head()

,dis_no,year,disaster_subgroup,disaster_type,disaster_subtype,event_name,country,region,continent,location,origin,associated_dis,dis_mag_value,dis_mag_scale,latitude,longitude,total_deaths,total_affected,start_date,end_date,duration,latitud_corregida,longitud_corregida
0,1970-0013-ARG,1970,Hydrological,Flood,Flood,NaN,Argentina,South America,Americas,Mendoza,NaN,NaN,NaN,Km2,NaN,NaN,36.0,NaN,1970-01-04,1970-01-04,1.0,NaN,NaN
1,1970-0109-AUS,1970,Meteorological,Storm,Tropical cyclone,Ada,Australia,Australia and New Zealand,Oceania,Queensland,NaN,NaN,NaN,Kph,NaN,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1970-0044-BEN,1970,Hydrological,Flood,Flood,NaN,Benin,Western Africa,Africa,Atacora region,NaN,NaN,NaN,Km2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1970-0063-BGD,1970,Meteorological,Storm,Tropical cyclone,NaN,Bangladesh,Southern Asia,Asia,"Khulna, Chittagong",NaN,NaN,NaN,Kph,NaN,NaN,300000.0,3648000.0,1970-11-12,1970-11-12,1.0,NaN,NaN
4,1970-0026-BGD,1970,Meteorological,Storm,Storm,NaN,Bangladesh,Southern Asia,Asia,Southern Asia,NaN,NaN,NaN,Kph,NaN,NaN,17.0,110.0,1970-04-13,1970-04-13,1.0,NaN,NaN


In [44]:
# pip install geopy


In [45]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

# --- Inicializar el geocodificador y un caché para las coordenadas ---
# El user_agent es un nombre único para tu aplicación.
geolocator = Nominatim(user_agent="corrector_desastres_v2")
location_cache = {}

def obtener_coordenadas(location_str, country_str):
    """
    Busca las coordenadas de una ubicación. Intenta primero con la localización
    específica y, si falla, usa el país como respaldo.
    """
    # Clave para el caché: combina localización y país para ser más preciso.
    cache_key = f"{location_str}, {country_str}".strip().lower()
    if cache_key in location_cache:
        return location_cache[cache_key]

    coords = None
    
    # 1. Intentar con la localización específica (ej. "Mendoza, Argentina")
    if pd.notna(location_str):
        try:
            full_location = f"{location_str}, {country_str}"
            # Pausa para no sobrecargar el servicio gratuito de geocodificación.
            time.sleep(1) 
            location_data = geolocator.geocode(full_location, language='es', timeout=10)
            if location_data:
                coords = (location_data.latitude, location_data.longitude)
        except Exception as e:
            print(f"  -> Advertencia al geocodificar '{full_location}': {e}")

    # 2. Si falla, intentar solo con el país como respaldo.
    if coords is None and pd.notna(country_str):
        print(f"  -> No se encontró '{location_str}'. Intentando con el país: '{country_str}'")
        try:
            time.sleep(1)
            location_data = geolocator.geocode(country_str, language='es', timeout=10)
            if location_data:
                coords = (location_data.latitude, location_data.longitude)
        except Exception as e:
            print(f"  -> Advertencia al geocodificar '{country_str}': {e}")

    # Guardar en caché el resultado (incluso si es None) para no repetir la búsqueda.
    location_cache[cache_key] = coords
    return coords

try:
    # --- Carga de datos ---
    # Usamos el archivo que ya tiene los títulos limpios.
    ruta_archivo = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_finales_limpios.csv'
    df = pd.read_csv(ruta_archivo)

    print("Iniciando la validación de coordenadas usando la columna 'location'...")
    
    filas_corregidas = 0
    # Iteramos sobre las filas que tienen coordenadas potencialmente inválidas para optimizar.
    # Un valor es inválido si no es un número o si está fuera del rango permitido.
    df_invalidos = df[
        (pd.to_numeric(df['latitud_corregida'], errors='coerce').isna()) |
        (df['latitud_corregida'].abs() > 90) |
        (pd.to_numeric(df['longitud_corregida'], errors='coerce').isna()) |
        (df['longitud_corregida'].abs() > 180)
    ]

    print(f"Se encontraron {len(df_invalidos)} filas con coordenadas potencialmente inválidas a revisar.")

    for index, row in df_invalidos.iterrows():
        lat = row['latitud_corregida']
        lon = row['longitud_corregida']
        
        # Comprobar si la latitud o longitud son inválidas
        lat_invalida = pd.notna(lat) and (lat > 90 or lat < -90)
        lon_invalida = pd.notna(lon) and (lon > 180 or lon < -180)

        # Solo proceder si realmente hay un error
        if lat_invalida or lon_invalida:
            location = row['location']
            country = row['country']
            
            print(f"\nFila {index}: Coordenadas inválidas (Lat: {lat}, Lon: {lon}).")
            print(f" -> Buscando corrección para: '{location}', '{country}'...")
            
            # Obtener coordenadas usando la nueva lógica (location > country)
            coords_nuevas = obtener_coordenadas(location, country)
            
            if coords_nuevas:
                lat_nueva, lon_nueva = coords_nuevas
                # Actualizar las columnas en el DataFrame original
                df.loc[index, 'latitud_corregida'] = lat_nueva
                df.loc[index, 'longitud_corregida'] = lon_nueva
                print(f" -> ¡Corregido! Nuevas coordenadas: (Lat: {lat_nueva:.4f}, Lon: {lon_nueva:.4f})")
                filas_corregidas += 1
            else:
                print(f" -> No se pudo encontrar una coordenada válida para '{location}, {country}'. Se mantendrá el valor original.")

    if filas_corregidas == 0:
        print("\nNo se encontraron coordenadas fuera de los rangos válidos que necesitaran corrección.")
    else:
        print(f"\nSe corrigieron un total de {filas_corregidas} filas con coordenadas inválidas.")

    # --- Guardado final ---
    ruta_salida = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\files\datos_finales_corregidos_v2.csv'
    df.to_csv(ruta_salida, index=False)
    
    print(f"\n¡Proceso finalizado! El archivo con las coordenadas corregidas y validadas se ha guardado en:\n{ruta_salida}")

except FileNotFoundError:
    print(f"Error: No se pudo encontrar el archivo en la ruta: {ruta_archivo}")
    print("Asegúrate de haber ejecutado el script anterior para generar 'datos_finales_limpios.csv'.")
except Exception as e:
    print(f"Ha ocurrido un error inesperado: {e}")


Iniciando la validación de coordenadas usando la columna 'location'...
Se encontraron 12381 filas con coordenadas potencialmente inválidas a revisar.

Fila 1275: Coordenadas inválidas (Lat: 24.688, Lon: 991.57).
 -> Buscando corrección para: 'Syhlet', 'Bangladesh'...
  -> No se encontró 'Syhlet'. Intentando con el país: 'Bangladesh'
 -> ¡Corregido! Nuevas coordenadas: (Lat: 24.4769, Lon: 90.2934)

Fila 2057: Coordenadas inválidas (Lat: 26.755, Lon: 616.0).
 -> Buscando corrección para: 'Dharbhanga, Madhubani, Saharsa, Munger, Khagana, Bihar Sharie (Bihar State)', 'India'...
  -> No se encontró 'Dharbhanga, Madhubani, Saharsa, Munger, Khagana, Bihar Sharie (Bihar State)'. Intentando con el país: 'India'
 -> ¡Corregido! Nuevas coordenadas: (Lat: 22.3511, Lon: 78.6677)

Fila 10616: Coordenadas inválidas (Lat: 125.091, Lon: 259.63).
 -> Buscando corrección para: 'Bohol district (Central Visayas province), Biliran, Eastern Samar, Leyte, Northern Samar, Samar, Southern Leyte districts (Easte

In [46]:
df_finales_corregidos = pd.read_csv("files/datos_finales_corregidos_v2.csv", index_col=0)
df_finales_corregidos = df_finales_corregidos.reset_index()
df_finales_corregidos.head()

,dis_no,year,disaster_subgroup,disaster_type,disaster_subtype,event_name,country,region,continent,location,origin,associated_dis,dis_mag_value,dis_mag_scale,latitude,longitude,total_deaths,total_affected,start_date,end_date,duration,latitud_corregida,longitud_corregida
0,1970-0013-ARG,1970,Hydrological,Flood,Flood,NaN,Argentina,South America,Americas,Mendoza,NaN,NaN,NaN,Km2,NaN,NaN,36.0,NaN,1970-01-04,1970-01-04,1.0,NaN,NaN
1,1970-0109-AUS,1970,Meteorological,Storm,Tropical cyclone,Ada,Australia,Australia and New Zealand,Oceania,Queensland,NaN,NaN,NaN,Kph,NaN,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1970-0044-BEN,1970,Hydrological,Flood,Flood,NaN,Benin,Western Africa,Africa,Atacora region,NaN,NaN,NaN,Km2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1970-0063-BGD,1970,Meteorological,Storm,Tropical cyclone,NaN,Bangladesh,Southern Asia,Asia,"Khulna, Chittagong",NaN,NaN,NaN,Kph,NaN,NaN,300000.0,3648000.0,1970-11-12,1970-11-12,1.0,NaN,NaN
4,1970-0026-BGD,1970,Meteorological,Storm,Storm,NaN,Bangladesh,Southern Asia,Asia,Southern Asia,NaN,NaN,NaN,Kph,NaN,NaN,17.0,110.0,1970-04-13,1970-04-13,1.0,NaN,NaN


In [47]:
# Estas 3 columnas aparecen con decimales cuando son deben ser números enteros. Las modificamos:
df_finales_corregidos["total_deaths"] = df_finales_corregidos["total_deaths"].astype("Int64")
df_finales_corregidos["total_affected"] = df_finales_corregidos["total_affected"].astype("Int64")
df_finales_corregidos["duration"] = df_finales_corregidos["duration"].astype("Int64")

In [48]:
df_finales_corregidos.head()

,dis_no,year,disaster_subgroup,disaster_type,disaster_subtype,event_name,country,region,continent,location,origin,associated_dis,dis_mag_value,dis_mag_scale,latitude,longitude,total_deaths,total_affected,start_date,end_date,duration,latitud_corregida,longitud_corregida
0,1970-0013-ARG,1970,Hydrological,Flood,Flood,NaN,Argentina,South America,Americas,Mendoza,NaN,NaN,NaN,Km2,NaN,NaN,36,<NA>,1970-01-04,1970-01-04,1,NaN,NaN
1,1970-0109-AUS,1970,Meteorological,Storm,Tropical cyclone,Ada,Australia,Australia and New Zealand,Oceania,Queensland,NaN,NaN,NaN,Kph,NaN,NaN,13,<NA>,NaN,NaN,<NA>,NaN,NaN
2,1970-0044-BEN,1970,Hydrological,Flood,Flood,NaN,Benin,Western Africa,Africa,Atacora region,NaN,NaN,NaN,Km2,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN
3,1970-0063-BGD,1970,Meteorological,Storm,Tropical cyclone,NaN,Bangladesh,Southern Asia,Asia,"Khulna, Chittagong",NaN,NaN,NaN,Kph,NaN,NaN,300000,3648000,1970-11-12,1970-11-12,1,NaN,NaN
4,1970-0026-BGD,1970,Meteorological,Storm,Storm,NaN,Bangladesh,Southern Asia,Asia,Southern Asia,NaN,NaN,NaN,Kph,NaN,NaN,17,110,1970-04-13,1970-04-13,1,NaN,NaN


In [49]:
df_finales_corregidos["longitud_corregida"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 14644 entries, 0 to 14643
Series name: longitud_corregida
Non-Null Count  Dtype  
--------------  -----  
2335 non-null   float64
dtypes: float64(1)
memory usage: 114.5 KB


In [50]:
df_finales_corregidos.to_csv("datos_finales_mapas.csv")

In [53]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time
from tqdm import tqdm # tqdm es una librería genial para mostrar barras de progreso

# --- CONFIGURACIÓN ---
# Asegúrate de que esta ruta sea la correcta para tu archivo
ruta_archivo_entrada = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_revisados.csv'
ruta_archivo_salida = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_con_coordenadas_reales.csv'

# 1. Cargar tus datos
# Usamos index_col=0 porque la primera columna en tu CSV es el índice antiguo
try:
    df = pd.read_csv(ruta_archivo_entrada, index_col=0)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {ruta_archivo_entrada}")
    # Salimos del script si el archivo no existe
    exit()

# 2. Preparar el geocodificador
# Es buena práctica establecer un 'user_agent' único para tu aplicación
geolocator = Nominatim(user_agent="proyecto_desastres_karen_1", timeout=10)

# 3. Crear la columna de búsqueda combinando 'location' y 'country'
# Rellenamos los valores nulos (NaN) en 'location' con un string vacío para evitar errores
df['Full Location'] = df['Location'].fillna('') + ', ' + df['Country']

# 4. Definir la función para obtener coordenadas de forma segura
def obtener_coordenadas_reales(direccion):
    """
    Obtiene latitud y longitud para una dirección.
    Maneja errores de conexión y timeouts.
    """
    # Si la dirección es inválida o solo contiene la coma, no hacemos la petición
    if pd.isna(direccion) or direccion.strip() == ',':
        return None, None
    
    try:
        # geocode() puede devolver None si no encuentra la dirección
        location = geolocator.geocode(direccion)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        print(f"Timeout para la dirección: {direccion}. Reintentando en 5 segundos...")
        time.sleep(5)
        return obtener_coordenadas_reales(direccion) # Reintento
    except GeocoderUnavailable:
        print(f"Servicio no disponible. Esperando 15 segundos para la dirección: {direccion}")
        time.sleep(15)
        return obtener_coordenadas_reales(direccion) # Reintento
    except Exception as e:
        print(f"Error inesperado para '{direccion}': {e}")
        return None, None

# 5. Aplicar la función a tu DataFrame
# tqdm nos mostrará una barra de progreso, muy útil para procesos largos.
# Hacemos una pausa de 1 segundo entre cada petición para no saturar el servicio.
tqdm.pandas(desc="Geocodificando localizaciones")

# Aplicamos la función y guardamos los resultados en dos nuevas columnas
df[['Latitud Real', 'Longitud Real']] = df['Full Location'].progress_apply(
    lambda loc: pd.Series(obtener_coordenadas_reales(loc))
)

# 6. Limpiar la columna auxiliar que creamos
df = df.drop(columns=['Full Location'])

# 7. Guardar el nuevo DataFrame en un archivo CSV
df.to_csv(ruta_archivo_salida)

# 8. Mostrar las primeras filas del resultado para verificar
print("\n¡Proceso completado!")
print(f"Los datos con las nuevas coordenadas se han guardado en: {ruta_archivo_salida}")
print("\nPrimeras 10 filas del DataFrame actualizado:")
print(df[['Location', 'Country', 'Latitud Real', 'Longitud Real']].head(10))

print("\nÚltimas 5 filas del DataFrame actualizado:")
print(df[['Location', 'Country', 'Latitud Real', 'Longitud Real']].tail(5))


Geocodificando localizaciones:   1%|▏         | 193/14644 [03:12<3:17:26,  1.22it/s]

Timeout para la dirección: Southwest of Western Australia, Australia. Reintentando en 5 segundos...
Timeout para la dirección: Southwest of Western Australia, Australia. Reintentando en 5 segundos...


Geocodificando localizaciones:   8%|▊         | 1191/14644 [20:41<3:39:06,  1.02it/s]

Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 segundos...
Timeout para la dirección: Northwest Buenos Aires province, Argentina. Reintentando en 5 se

Geocodificando localizaciones:  55%|█████▍    | 8027/14644 [14:49:46<646:48:35, 351.90s/it]   

: 

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time
from tqdm import tqdm

# --- CONFIGURACIÓN ---
# Asegúrate de que estas rutas sean las correctas para tu archivo
ruta_archivo_entrada = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_revisados.csv'
ruta_archivo_salida = r'c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_con_coordenadas_reales.csv'

# 1. Cargar tus datos
print("Cargando el archivo CSV...")
try:
    # Usamos index_col=0 porque la primera columna en tu CSV es el índice antiguo
    df = pd.read_csv(ruta_archivo_entrada, index_col=0)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {ruta_archivo_entrada}")
    exit()

# 2. Preparar el geocodificador
# Es buena práctica establecer un 'user_agent' único para tu aplicación
geolocator = Nominatim(user_agent="proyecto_desastres_karen_adalab_v2", timeout=15)

# 3. Optimización: Trabajar solo con localizaciones únicas
print("Identificando localizaciones únicas para optimizar las peticiones...")
# Combinamos 'Location' y 'Country' para crear una dirección completa y única
df['Full Location'] = df['Location'].fillna('') + ', ' + df['Country']
localizaciones_unicas = df['Full Location'].unique()

print(f"Se han encontrado {df.shape[0]} filas en total.")
print(f"Se procesarán {len(localizaciones_unicas)} localizaciones únicas en lugar de una por cada fila.")

# 4. Crear un diccionario para almacenar las coordenadas obtenidas
coordenadas_cache = {}

# 5. Iterar sobre las localizaciones únicas con una barra de progreso (tqdm)
print("\nIniciando proceso de geocodificación...")
for direccion in tqdm(localizaciones_unicas, desc="Geocodificando"):
    # Si la dirección es inválida o solo contiene la coma, la saltamos
    if pd.isna(direccion) or direccion.strip() == ',':
        coordenadas_cache[direccion] = (None, None)
        continue

    try:
        # Hacemos la petición para obtener la geolocalización
        location = geolocator.geocode(direccion)
        if location:
            coordenadas_cache[direccion] = (location.latitude, location.longitude)
        else:
            coordenadas_cache[direccion] = (None, None)
            
        # IMPORTANTE: Pausa de 1 segundo para respetar los límites de la API de Nominatim
        time.sleep(1)

    except (GeocoderTimedOut, GeocoderUnavailable) as e:
        print(f"Error de servicio para '{direccion}': {e}. Se reintentará más tarde si es necesario. Marcando como nulo por ahora.")
        coordenadas_cache[direccion] = (None, None)
    except Exception as e:
        print(f"Error inesperado para '{direccion}': {e}")
        coordenadas_cache[direccion] = (None, None)

# 6. Mapear los resultados del diccionario de vuelta al DataFrame
print("\nMapeando las coordenadas obtenidas al DataFrame principal...")
# Usamos el método .map() que es muy eficiente para esta tarea
coordenadas_series = df['Full Location'].map(coordenadas_cache)

# Asignamos los resultados a las nuevas columnas
df['Latitud Real'] = coordenadas_series.str[0]
df['Longitud Real'] = coordenadas_series.str[1]

# 7. Limpiar la columna auxiliar que creamos
df = df.drop(columns=['Full Location'])

# 8. Guardar el nuevo DataFrame en un archivo CSV
print(f"\nGuardando los resultados en: {ruta_archivo_salida}")
df.to_csv(ruta_archivo_salida)

# 9. Mostrar información final y verificar
print("\n¡Proceso completado!")
filas_con_coords = df['Latitud Real'].notna().sum()
print(f"Se obtuvieron coordenadas para {filas_con_coords} de {df.shape[0]} filas.")

print("\nPrimeras 10 filas del DataFrame actualizado:")
print(df[['Location', 'Country', 'Latitud Real', 'Longitud Real']].head(10))

print("\nÚltimas 5 filas del DataFrame actualizado:")
print(df[['Location', 'Country', 'Latitud Real', 'Longitud Real']].tail(5))


Cargando el archivo CSV...
Identificando localizaciones únicas para optimizar las peticiones...
Se han encontrado 14644 filas en total.
Se procesarán 12418 localizaciones únicas en lugar de una por cada fila.

Iniciando proceso de geocodificación...


Geocodificando: 100%|██████████| 12418/12418 [4:31:13<00:00,  1.31s/it]  



Mapeando las coordenadas obtenidas al DataFrame principal...

Guardando los resultados en: c:\Users\krigu\OneDrive\Desktop\ADALAB\Modulo-4\Proyecto_desastres_naturales_modulo_4\datos_con_coordenadas_reales.csv

¡Proceso completado!
Se obtuvieron coordenadas para 2768 de 14644 filas.

Primeras 10 filas del DataFrame actualizado:
                   Location     Country  Latitud Real  Longitud Real
0                   Mendoza   Argentina    -34.787093     -68.438187
1                Queensland   Australia    -22.164678     144.584490
2            Atacora region       Benin           NaN            NaN
3        Khulna, Chittagong  Bangladesh           NaN            NaN
4             Southern Asia  Bangladesh           NaN            NaN
5             Southern Asia  Bangladesh           NaN            NaN
6   Bay of Bengal districts  Bangladesh           NaN            NaN
7                 Northeast      Brazil     -3.747784     -38.461060
8  Recife, South Pernambuco      Brazil         

Al cargar en Tableau nos ha modificado columnas y éstas no aparecen correctamente.

Dejamos estas pruebas para next steps